In [0]:
%run ./01_logging_utils.py

In [0]:

from datetime import datetime
from pyspark.sql import functions as F


def silver_transform():
    run_id = new_batch_id("silver")
    start_time = datetime.now()

    try:
        # Read Bronze data
        bronze_df = spark.table(T_BRONZE)

        # Convert timestamp from milliseconds to timestamp
        transformed_df = (
            bronze_df
            .withColumn(
                "event_ts",
                F.to_timestamp(
                    F.from_unixtime(F.col("ts_ms") / 1000.0)
                )
            )
            .withColumn("event_date", F.to_date("event_ts"))
            .withColumn("event_hour", F.hour("event_ts"))
        )

        # Separate valid records from invalid records
        valid_df = transformed_df.filter(
            F.col("coin_id").isNotNull()
            & F.col("ts_ms").isNotNull()
            & F.col("event_ts").isNotNull()
            & F.col("price").isNotNull()
            & (F.col("price") >= 0)
            & (
                F.col("market_cap").isNull()
                | (F.col("market_cap") >= 0)
            )
            & (
                F.col("total_volume").isNull()
                | (F.col("total_volume") >= 0)
            )
        )

        # Match the Silver table's column order
        valid_df = (
            valid_df
            .select(
                "coin_id",
                "event_ts",
                "price",
                "market_cap",
                "total_volume",
                "event_date",
                "event_hour",
                "batch_id",
                "load_timestamp"
            )
            .dropDuplicates(["coin_id", "event_ts"])
        )

        # Merge records to make reruns safe
        valid_df.createOrReplaceTempView("silver_source")

        spark.sql(f"""
            MERGE INTO {T_SILVER} AS target
            USING silver_source AS source
            ON target.coin_id = source.coin_id
               AND target.event_ts = source.event_ts

            WHEN MATCHED AND NOT (
                target.price <=> source.price
                AND target.market_cap <=> source.market_cap
                AND target.total_volume <=> source.total_volume
                AND target.event_date <=> source.event_date
                AND target.event_hour <=> source.event_hour
            )
            THEN UPDATE SET *

            WHEN NOT MATCHED THEN INSERT *
        """)

        inserted, updated = merge_metrics(T_SILVER)

        log_run(
            run_id,
            "silver",
            "Bronze to Silver transformation",
            start_time,
            datetime.now(),
            "SUCCESS",
            inserted,
            updated
        )

        print("Silver transformation completed successfully.")
        print(f"Valid rows processed: {valid_df.count()}")
        print(f"Rows inserted: {inserted}")
        print(f"Rows updated: {updated}")

    except Exception as e:
        log_run(
            run_id,
            "silver",
            "Bronze to Silver transformation",
            start_time,
            datetime.now(),
            "FAILED",
            err=str(e)[:500]
        )
        raise


In [0]:
silver_transform()

verifying silver table

In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT coin_id) AS distinct_coins,
        MIN(event_ts) AS earliest_timestamp,
        MAX(event_ts) AS latest_timestamp
    FROM crypto.silver.market_data_clean
"""))

**checking for duplicate coin/timestamp format**

In [0]:
display(spark.sql("""
    SELECT coin_id, event_ts, COUNT(*) AS record_count
    FROM crypto.silver.market_data_clean
    GROUP BY coin_id, event_ts
    HAVING COUNT(*) > 1
"""))